<a href="https://colab.research.google.com/github/startupmini/BaseModel/blob/main/laya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/convaiinnovations/laya

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/convaiinnovations/laya)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [22]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="convaiinnovations/laya")

ValueError: Unrecognized model in convaiinnovations/laya. Should have a `model_type` key in its config.json.

In [ ]:
# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("convaiinnovations/laya", device_map="auto")

### Persiapan Library untuk Fine-Tuning yang Aman (PEFT/QLoRA)
Untuk melakukan *fine-tuning* model tanpa kehabisan memori (OOM) di GPU gratis Colab, jalankan instalasi pustaka pendukung berikut:

In [ ]:
!pip install -q -U bitsandbytes peft accelerate trl sentencepiece tiktoken

In [ ]:
from huggingface_hub import HfApi

# Ambil info detail model dari Hugging Face Hub
api = HfApi()
try:
    model_info = api.model_info("convaiinnovations/laya")
    print("Model ID:", model_info.modelId)
    print("Tags/Task:", model_info.tags)
    print("Pipeline Tag:", getattr(model_info, "pipeline_tag", "Tidak diketahui"))
except Exception as e:
    print("Gagal mengambil metadata:", e)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

print("=== Mencari Dataset Crypto / Forex untuk Text Classification ===\n")

# Cari dataset bertema crypto, forex, atau finance
queries = ["crypto sentiment", "forex sentiment", "financial sentiment"]

for query in queries:
    print(f"Hasil pencarian untuk: '{query}'")
    datasets = api.list_datasets(search=query, limit=3)
    for ds in datasets:
        print(f"- Dataset ID: {ds.id}")
        print(f"  Downloads: {getattr(ds, 'downloads', 0)}")
        print(f"  Likes: {getattr(ds, 'likes', 0)}\n")

In [ ]:
from datasets import load_dataset

print("Memuat dataset sentimen keuangan...")
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")

print("\nStruktur Dataset:")
print(dataset)

print("\nContoh 3 data pertama dari training set:")
for i in range(3):
    print(f"- Teks: {dataset['train'][i]['text']}")
    print(f"  Label: {dataset['train'][i]['label']} (0: Bearish/Negatif, 1: Bullish/Positif, 2: Netral)\n")

In [ ]:
!pip install -q yfinance pandas

### Mengambil Data Historis XAU/USD (Gold) & BTC/USD (Bitcoin)
Sekarang kita buat fungsi untuk menarik data pasar historis yang bisa kita sinkronkan dengan dataset sentimen atau digunakan untuk melatih model klasifikasi berbasis pergerakan harga.

In [ ]:
import yfinance as yf
import pandas as pd

def ambil_data_market(ticker, nama_aset, start_date="2023-01-01", end_date="2024-01-01"):
    print(f"Mengambil data historis {nama_aset} ({ticker})...")
    data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
    data = data.reset_index()
    # Hitung persentase perubahan harian untuk klasifikasi (Naik = 1, Turun = 0)
    # Menggunakan sintaks yang kompatibel dengan multi-index columns dari yfinance baru
    close_col = data['Close']
    if isinstance(close_col, pd.DataFrame):
        close_col = close_col.iloc[:, 0]

    data['Return'] = close_col.pct_change()
    data['Label_Arah'] = data['Return'].apply(lambda x: 1 if x > 0 else 0)
    return data

# Ambil data Emas (XAU/USD diwakili oleh GC=F) dan Bitcoin (BTC-USD)
df_gold = ambil_data_market("GC=F", "Emas (XAU/USD)")
df_btc = ambil_data_market("BTC-USD", "Bitcoin (BTC/USD)")

print("\nContoh Data XAU/USD:")
display(df_gold.head())

print("\nContoh Data BTC/USD:")
display(df_btc.head())

### Menyiapkan Fine-Tuning Laya dengan PEFT (LoRA)
Di bawah ini adalah skrip untuk memuat model Laya, mengonfigurasi LoRA, dan menyiapkan Trainer untuk melatih model menggunakan dataset sentimen keuangan.

In [ ]:
import torch
from transformers import LlamaConfig, LlamaTokenizer, LlamaForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model_id = "convaiinnovations/laya"

# 1. Load Tokenizer menggunakan kelas LlamaTokenizer
print("Loading Tokenizer...")
tokenizer = LlamaTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Load Config secara spesifik menggunakan LlamaConfig untuk bypass masalah model_type
print("Loading Configuration...")
config = LlamaConfig.from_pretrained(
    model_id,
    trust_remote_code=True,
    num_labels=3
)

# 3. Load Model menggunakan kelas LlamaForSequenceClassification secara eksplisit
print("Loading Model...")
model = LlamaForSequenceClassification.from_pretrained(
    model_id,
    config=config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# 4. Konfigurasi LoRA agar Hemat Memori
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("\nModel Laya berhasil dimuat menggunakan arsitektur LLaMA dan siap untuk proses fine-tuning!")

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

# 1. Bersihkan dataset dari baris teks kosong atau invalid
print("Membersihkan teks kosong...")
clean_dataset = dataset.filter(lambda example: example['text'] is not None and len(example['text'].strip()) > 5)

# 2. Fungsi Tokenisasi untuk dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

print("Melakukan preprocessing & tokenisasi pada dataset...")
# Tokenisasi dataset yang telah dibersihkan
tokenized_dataset = clean_dataset.map(preprocess_function, batched=True)

# Pastikan format dataset sesuai untuk PyTorch
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# Data collator untuk padding dinamis
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3. Definisikan Argumen Pelatihan (Training Arguments)
training_args = TrainingArguments(
    output_dir="./results_laya_finance",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    fp16=True,
    report_to="none"
)

# 4. Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

# 5. Jalankan Fine-Tuning
print("Memulai proses fine-tuning model Laya...")
trainer.train()

print("\nFine-tuning selesai dengan sukses!")

In [ ]:
import torch
from transformers import LlamaTokenizer, LlamaForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

model_id = "convaiinnovations/laya"

# 1. Bersihkan dan muat ulang dataset
print("Memuat dan membersihkan dataset...")
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
clean_dataset = dataset.filter(lambda x: x['text'] is not None and len(x['text'].strip()) > 5)

# 2. Muat Tokenizer
print("Memuat Tokenizer...")
tokenizer = LlamaTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 3. Tokenisasi Dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_dataset = clean_dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# 4. Muat Model Klasifikasi Laya
print("Memuat Model...")
model = LlamaForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model.config.pad_token_id = tokenizer.pad_token_id

# 5. Konfigurasi LoRA untuk Keamanan Memori (VRAM)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 6. Definisikan Argumen Pelatihan dan Trainer
training_args = TrainingArguments(
    output_dir="./results_laya_finance",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

# 7. Jalankan Fine-Tuning
print("Memulai fine-tuning...")
trainer.train()
print("Fine-tuning berhasil selesai!")

Memuat dan membersihkan dataset...


Filter:   0%|          | 0/9543 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2388 [00:00<?, ? examples/s]

Memuat Tokenizer...


Map:   0%|          | 0/9522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2383 [00:00<?, ? examples/s]

Memuat Model...


Loading weights: 0it [00:00, ?it/s]

In [2]:
# Jalankan sel ini setelah proses training selesai untuk melihat riwayat loss & evaluasi secara rapi
import pandas as pd

try:
    history_df = pd.DataFrame(trainer.state.log_history)
    print("=== Riwayat Metrik Pelatihan (Loss & Eval) ===")
    display(history_df.dropna(subset=['loss', 'eval_loss'], how='all'))
except NameError:
    print("Pastikan Anda telah berhasil menyelesaikan eksekusi sel pelatihan (15174a0d) terlebih dahulu.")

Pastikan Anda telah berhasil menyelesaikan eksekusi sel pelatihan (15174a0d) terlebih dahulu.


In [1]:
# Visualisasi Real-Time / Pasca-Pelatihan untuk Perkembangan Loss
import matplotlib.pyplot as plt
import pandas as pd
import time

try:
    # Ambil riwayat log
    logs = trainer.state.log_history

    # Pisahkan metrik training dan evaluation
    train_loss = [x for x in logs if 'loss' in x]
    eval_loss = [x for x in logs if 'eval_loss' in x]

    df_train = pd.DataFrame(train_loss)
    df_eval = pd.DataFrame(eval_loss)

    # Plotting
    plt.figure(figsize=(10, 5))

    if not df_train.empty:
        plt.plot(df_train['step'], df_train['loss'], label='Training Loss', marker='o', color='blue')
    if not df_eval.empty:
        plt.plot(df_eval['step'], df_eval['eval_loss'], label='Evaluation Loss', marker='x', color='red')

    plt.title('Grafik Progress Fine-Tuning Laya (Loss)')
    plt.xlabel('Steps / Iterasi')
    plt.ylabel('Loss Value')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()

    print(f"Status: Berhasil memuat {len(df_train)} log training dan {len(df_eval)} log evaluasi.")

except NameError:
    print("INFO: Sel pelatihan '15174a0d' belum selesai dijalankan atau sedang aktif melakukan inisialisasi awal.")
    print("Silakan tunggu hingga baris 'Memulai fine-tuning...' di sel atas mulai memproses langkah pertama (Step 10/20/dst) lalu jalankan kembali sel ini.")

INFO: Sel pelatihan '15174a0d' belum selesai dijalankan atau sedang aktif melakukan inisialisasi awal.
Silakan tunggu hingga baris 'Memulai fine-tuning...' di sel atas mulai memproses langkah pertama (Step 10/20/dst) lalu jalankan kembali sel ini.


In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
from transformers import LlamaTokenizer, LlamaForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from IPython.display import clear_output, display

# 1. Custom Callback untuk Visualisasi Real-Time saat training berjalan
class VisualProgressCallback(TrainerCallback):
    def __init__(self):
        self.steps = []
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.steps.append(state.global_step)
            self.losses.append(logs["loss"])

            # Bersihkan output layar lama untuk memperbarui grafik
            clear_output(wait=True)

            # Gambar ulang grafik perkembangan
            plt.figure(figsize=(10, 5))
            plt.plot(self.steps, self.losses, label='Training Loss', marker='o', color='#1f77b4', linewidth=2)
            plt.title('Visualisasi Real-Time Progress Fine-Tuning Laya (Loss)', fontsize=14, fontweight='bold')
            plt.xlabel('Steps (Iterasi)', fontsize=12)
            plt.ylabel('Loss Value', fontsize=12)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.legend(fontsize=11)
            plt.show()

            print(f"[Step {state.global_step}] Loss Terbaru: {logs['loss']:.4f}")

# 2. Muat ulang dan siapkan data
print("1. Memuat dan membersihkan dataset...")
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
clean_dataset = dataset.filter(lambda x: x['text'] is not None and len(x['text'].strip()) > 5)

print("2. Memuat Tokenizer Laya...")
model_id = "convaiinnovations/laya"
tokenizer = LlamaTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

print("3. Melakukan tokenisasi dataset...")
tokenized_dataset = clean_dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# 3. Muat Model Klasifikasi
print("4. Memuat model Laya dengan GPU...")
model = LlamaForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # Wajib dinonaktifkan untuk training / fine-tuning

# 4. Terapkan LoRA PEFT
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. Konfigurasi Training
training_args = TrainingArguments(
    output_dir="./results_laya_finance",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,  # Update grafik setiap 10 steps
    fp16=True,
    report_to="none"
)

# 6. Inisialisasi Trainer bersama Custom Callback
visual_callback = VisualProgressCallback()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    callbacks=[visual_callback]  # Pasang callback visual di sini
)

# 7. Mulai pelatihan dengan visualisasi aktif
print("5. Memulai proses fine-tuning dengan monitor visual...")
trainer.train()
print("Fine-tuning selesai dengan sukses!")

1. Memuat dan membersihkan dataset...
2. Memuat Tokenizer Laya...
3. Melakukan tokenisasi dataset...
4. Memuat model Laya dengan GPU...


Loading weights: 0it [00:00, ?it/s]